<a href="https://colab.research.google.com/github/nejumi/models-initial-course/blob/main/notebooks/00_setup/01_segmentation_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# まずは道路画像を学習して、予測を見てみよう

道路・車・空などをピクセルごとに塗り分ける「セグメンテーション」を試します。
小さな U-Net を2エポック学習させ、loss の推移と予測画像を W&B に記録します。
最初はセルを順に実行し、画像を見るところまで進めましょう。API の書き方は次の基礎編で説明します。

CPUでも実行できます。CUDA GPU が使える場合は自動で利用します。
初回はパッケージ、公開データ、ResNet18 の学習済み重みをダウンロードするため、インターネット接続が必要です。
接続先の設定は環境セットアップと同じです。Dedicated Cloud でも同じコードで実行します。

In [ ]:
# リポジトリ内ではルートを使い、Colab 単体では共通コードを取得します。
from pathlib import Path
import os
import sys
import urllib.request

root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "course_utils.py").is_file()), Path.cwd())
if not (root / "course_utils.py").exists():
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/nejumi/models-initial-course/main/course_utils.py",
        root / "course_utils.py",
    )
os.chdir(root)
sys.path.insert(0, str(root))

from course_utils import setup_wandb
PROJECT, ENTITY, WANDB_INIT = setup_wandb(packages=['fastai>=2.8.0,<3', 'torchvision>=0.16.0'])

## 1. 学習する画像を用意する

[fastai が配布する CamVid tiny](https://docs.fast.ai/tutorial.vision.html#segmentation) の道路画像と正解マスクを使います。
各色が道路・車・空などのクラスを表します。まず64枚を使い、約2割を学習に使わない検証用に分けます。

短時間で記録と可視化を体験するための設定です。精度を評価するための本格的な実験ではありません。

In [ ]:
from pathlib import Path
import numpy as np
import torch
from fastai.vision.all import (
    URLs, untar_data, get_image_files, set_seed,
    SegmentationDataLoaders, Resize, Normalize, imagenet_stats,
    unet_learner, resnet18,
)
from fastai.vision.all import Callback, Recorder
import wandb

config = dict(epochs=2, lr=2e-3, batch_size=4, max_images=64,
              image_size=[96, 128], seed=42, pretrained=True)
set_seed(config["seed"], reproducible=True)
torch.set_num_threads(min(4, torch.get_num_threads()))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("使用するデバイス:", device)

path = untar_data(URLs.CAMVID_TINY, base=Path("outputs/data").resolve())
codes = np.loadtxt(path / "codes.txt", dtype=str)
files = sorted(get_image_files(path / "images"))
rng = np.random.default_rng(config["seed"])
files = [files[i] for i in rng.permutation(len(files))[:config["max_images"]]]

def label_path(image_path):
    return path / "labels" / f"{image_path.stem}_P{image_path.suffix}"

dls = SegmentationDataLoaders.from_label_func(
    path, files, label_path, codes=codes,
    valid_pct=0.2, seed=config["seed"],
    bs=config["batch_size"], num_workers=0, device=device,
    item_tfms=Resize(tuple(config["image_size"])),
    batch_tfms=Normalize.from_stats(*imagenet_stats),
)
print(f"学習: {len(dls.train_ds)} 枚 / 検証: {len(dls.valid_ds)} 枚")
dls.show_batch(max_n=4, figsize=(10, 6))

## 2. 学習し、W&B に記録する

小さな callback を用意して、バッチごとの loss とエポックごとの評価値を自動で記録します。
学習後は同じ検証バッチから画像・正解・予測を取り出して、W&B の Table に並べます。
`Void`（未ラベル）のピクセルは loss と accuracy の計算から除きます。

In [ ]:
from fastai.vision.all import CrossEntropyLossFlat

class LogToWandb(Callback):
    # Recorder が評価値を計算した後に呼び出します。
    order = Recorder.order + 1

    def after_batch(self):
        if self.training:
            wandb.log({"batch/train_loss": self.loss.item()})

    def after_epoch(self):
        metrics = dict(zip(self.recorder.metric_names[1:-1], self.recorder.values[-1]))
        wandb.log({"epoch": self.epoch + 1, **metrics})

void_id = int(np.where(codes == "Void")[0][0])

def pixel_accuracy(prediction, target):
    valid = target != void_id
    return (prediction.argmax(dim=1)[valid] == target[valid]).float().mean()

learn = unet_learner(
    dls, resnet18, pretrained=config["pretrained"],
    loss_func=CrossEntropyLossFlat(axis=1, ignore_index=void_id),
    metrics=pixel_accuracy,
    path=Path("outputs/segmentation"),
)
class_labels = dict(enumerate(codes.tolist()))

with wandb.init(**WANDB_INIT, name="00-road-segmentation",
                config={**config, "device": str(device)},
                tags=["course", "segmentation"]) as run:
    learn.fit_one_cycle(
        config["epochs"], config["lr"],
        cbs=LogToWandb(),
    )

    # 同じバッチを使うことで、画像・正解・予測の順番を一致させます。
    images, targets = dls.valid.one_batch()
    learn.model.eval()
    with torch.no_grad():
        predictions = learn.model(images).argmax(dim=1).cpu().numpy()
    mean = torch.tensor(imagenet_stats[0], device=device)[None, :, None, None]
    std = torch.tensor(imagenet_stats[1], device=device)[None, :, None, None]
    rgb_images = ((images * std + mean).clamp(0, 1) * 255).byte()
    rgb_images = rgb_images.permute(0, 2, 3, 1).cpu().numpy()
    targets = targets.cpu().numpy()

    table = wandb.Table(columns=["sample", "image", "ground_truth_and_prediction"])
    for index, (rgb, target, prediction) in enumerate(zip(rgb_images, targets, predictions)):
        overlay = wandb.Image(rgb, masks={
            "ground_truth": {"mask_data": target, "class_labels": class_labels},
            "prediction": {"mask_data": prediction, "class_labels": class_labels},
        })
        table.add_data(index, wandb.Image(rgb), overlay)
    run.log({"predictions": table})
    run.summary["validation_images"] = len(dls.valid_ds)
    run_url = run.url

print("Run URL:", run_url)

## 3. 予測を見てみる

Notebook でも同じ画像を並べます。左から入力画像・正解・予測です。
W&B の run を開き、`train_loss` と `valid_loss` のグラフ、`predictions` の Table を確認してください。
Table の画像では `ground_truth` と `prediction` のマスクを切り替えられます。

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(rgb_images), 3, figsize=(10, 3 * len(rgb_images)), squeeze=False)
for row, (rgb, target, prediction) in enumerate(zip(rgb_images, targets, predictions)):
    axes[row, 0].imshow(rgb)
    axes[row, 1].imshow(target, vmin=0, vmax=len(codes) - 1, cmap="tab20")
    axes[row, 2].imshow(prediction, vmin=0, vmax=len(codes) - 1, cmap="tab20")
    for ax, title in zip(axes[row], ["Image", "Ground truth", "Prediction"]):
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
Path("outputs/segmentation").mkdir(parents=True, exist_ok=True)
fig.savefig("outputs/segmentation/predictions.png", dpi=120)
plt.show()

道路や空の形は捉えられましたか？ 細かい物体や画像の境界で、正解と予測を見比べてみてください。
時間があれば `epochs` を5にして、データ準備のセルから再実行します。新しい run と比較できます。

次は [基本の実験記録](../01_basics/01_minimal_tracking.ipynb) へ進み、
いま使った `wandb.init`、`config`、`run.log` を小さな例で確認します。

### 出典と変更点

「W&B Monthly Segmentation」（W&B Courses から一部改変・抜粋）をもとに、
fastai の U-Net 学習・W&B 連携・予測マスクの比較という流れを残しました。
特定アカウントの BDD Artifact は公開 CamVid tiny に置き換え、学習枚数・エポック数を減らしています。
古い固定バージョンの一括インストール、未使用の評価クラス、外部の `params.py` への参照を除きました。

- [W&B の MLOps 講座](https://github.com/wandb/edu/tree/main/mlops-001)
- [fastai のセグメンテーション例](https://docs.fast.ai/tutorial.vision.html#segmentation)
- [CamVid データセットの出典](http://mi.eng.cam.ac.uk/research/projects/VideoRec/CamVid/)
